# P-016 Novel Composition Holdout Simulation
## Agent OS v3.0 | Materia Arche

**Decision question:** Can we promise a lab partner that our model will rank their novel compositions correctly, or only compositions similar to what we've trained on?

**Method:** Leave-One-Family-Out (LOFO) cross-validation. For each composition family, we train on all *other* families and predict on the held-out family. This simulates a lab synthesising a composition family the model has never seen.

**Panel:** Device 850 (MA-only, T80=3400h), Device 1320 (MA-FA mixed, T80=940h), Device 119 (FA-Cs, T80=3423h)

**Prior result (P-013):** LOGO CV dropped tau-b from 0.289 to 0.055 on composition clusters.

In [1]:
"""Cell 2 — Load data and classify composition families."""
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from sklearn.ensemble import ExtraTreesRegressor

# ── Load ──
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {len(df)} devices, {df.shape[1]} columns")

# ── Features & target ──
FEATURES = [
    "Perovskite_band_gap", "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature", "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness", "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF",
]
TARGET = "Stability_PCE_T80"

# FIX: use fillna(0) for features instead of dropna() — consistent with all other notebooks
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f"After dropping missing TARGET: {len(df)} devices")

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])
df["log_T80"] = y

# ── Classify composition family (on original df, not a dropna'd version) ──
def classify_family(row):
    ma = row["MA"] > 0
    fa = row["FA"] > 0
    cs = row["Cs"] > 0
    if ma and not fa and not cs:
        return "Pure MA"
    elif fa and not ma and not cs:
        return "Pure FA"
    elif ma and fa and not cs:
        return "MA-FA mixed"
    elif fa and cs and not ma:
        return "FA-Cs"
    elif ma and fa and cs:
        return "Triple cation"
    else:
        return "Other"

df["family"] = df.apply(classify_family, axis=1)
family_counts = df["family"].value_counts()
print("\n── Composition family counts ──")
print(family_counts.to_string())
print(f"\nTotal: {family_counts.sum()}")

Dataset: 1543 devices, 47 columns
After dropping missing TARGET: 1543 devices

── Composition family counts ──
family
Pure MA          967
Triple cation    197
MA-FA mixed      197
Other             88
Pure FA           50
FA-Cs             44

Total: 1543


In [2]:
"""Cell 3 — Leave-One-Family-Out (LOFO) cross-validation."""

ET_PARAMS = dict(
    n_estimators=700,
    max_features="sqrt",
    min_samples_split=3,
    min_samples_leaf=1,
    bootstrap=False,
    random_state=42,
)

lofo_results = []
families_tested = []

for family_name in sorted(df["family"].unique()):
    mask_test = df["family"] == family_name
    n_test = mask_test.sum()
    
    if n_test < 20:
        print(f"  SKIP {family_name}: only {n_test} devices (< 20)")
        continue
    
    X_train = df.loc[~mask_test, FEATURES].fillna(0).values
    y_train = df.loc[~mask_test, "log_T80"].values
    X_test  = df.loc[mask_test, FEATURES].fillna(0).values
    y_test  = df.loc[mask_test, "log_T80"].values
    
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    tau, p_val = kendalltau(y_test, y_pred)
    
    lofo_results.append({
        "family": family_name,
        "n_devices": n_test,
        "n_train": len(y_train),
        "lofo_tau_b": round(tau, 4),
        "p_value": round(p_val, 4),
    })
    families_tested.append(family_name)
    
    print(f"  {family_name:20s}  n={n_test:4d}  tau-b={tau:+.4f}  p={p_val:.4f}")

lofo_df = pd.DataFrame(lofo_results)
mean_tau = lofo_df["lofo_tau_b"].mean()
print(f"\n── Mean LOFO tau-b: {mean_tau:.4f} ──")
print(f"── Families tested: {len(families_tested)} ──")

  FA-Cs                 n=  44  tau-b=-0.0488  p=0.6416


  MA-FA mixed           n= 197  tau-b=-0.0724  p=0.1332


  Other                 n=  88  tau-b=+0.0566  p=0.4382


  Pure FA               n=  50  tau-b=+0.1420  p=0.1476


  Pure MA               n= 967  tau-b=+0.0783  p=0.0003


  Triple cation         n= 197  tau-b=-0.1264  p=0.0087

── Mean LOFO tau-b: 0.0049 ──
── Families tested: 6 ──


In [3]:
"""Cell 4 — Panel family simulation."""

PANEL = {850: "MA-only", 1320: "MA-FA mixed", 119: "FA-Cs"}

print("── Panel device family simulation ──\n")

for dev_idx, label in PANEL.items():
    if dev_idx >= len(df):
        print(f"  Device {dev_idx} ({label}): index out of range, skipping")
        continue
    
    row = df.iloc[dev_idx]
    fam = row["family"]
    actual_t80 = row[TARGET]
    
    # Find LOFO tau-b for this family
    lofo_row = lofo_df[lofo_df["family"] == fam]
    if len(lofo_row) == 0:
        print(f"  Device {dev_idx} ({label}): family '{fam}' was skipped (< 20 devices)")
        continue
    
    family_tau = lofo_row["lofo_tau_b"].values[0]
    
    # Train model without this family, predict on held-out family
    mask_fam = df["family"] == fam
    X_train = df.loc[~mask_fam, FEATURES].fillna(0).values
    y_train = df.loc[~mask_fam, "log_T80"].values
    X_test  = df.loc[mask_fam, FEATURES].fillna(0).values
    y_test  = df.loc[mask_fam, "log_T80"].values
    
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Find rank of the panel device within its family
    fam_indices = df.index[mask_fam].tolist()
    dev_pos = fam_indices.index(dev_idx)
    dev_pred = y_pred[dev_pos]
    
    # Rank by predicted value (descending = higher predicted stability is rank 1)
    pred_ranks = (-y_pred).argsort().argsort() + 1  # 1-based rank
    dev_rank = pred_ranks[dev_pos]
    n_family = mask_fam.sum()
    top20 = dev_rank <= 20
    
    print(f"  Device {dev_idx} ({label})")
    print(f"    Family: {fam}  |  LOFO tau-b: {family_tau:+.4f}")
    print(f"    Actual T80: {actual_t80:.0f}h  |  Predicted rank: {dev_rank}/{n_family}")
    print(f"    In top-20 of held-out family? {'YES' if top20 else 'NO'}")
    print()

── Panel device family simulation ──



  Device 850 (MA-only)
    Family: Pure MA  |  LOFO tau-b: +0.0783
    Actual T80: 3400h  |  Predicted rank: 440/967
    In top-20 of held-out family? NO



  Device 1320 (MA-FA mixed)
    Family: MA-FA mixed  |  LOFO tau-b: -0.0724
    Actual T80: 940h  |  Predicted rank: 83/197
    In top-20 of held-out family? NO



  Device 119 (FA-Cs)
    Family: FA-Cs  |  LOFO tau-b: -0.0488
    Actual T80: 3423h  |  Predicted rank: 15/44
    In top-20 of held-out family? YES



In [4]:
"""Cell 5 — Compare LOFO tau-b vs random CV tau-b."""

RANDOM_CV_TAU = 0.289

comparison = lofo_df[["family", "n_devices", "lofo_tau_b", "p_value"]].copy()
comparison["random_cv_tau_b"] = RANDOM_CV_TAU
comparison["gap"] = comparison["random_cv_tau_b"] - comparison["lofo_tau_b"]
comparison["pct_retained"] = (comparison["lofo_tau_b"] / RANDOM_CV_TAU * 100).round(1)

print("── LOFO tau-b vs Random CV tau-b (0.289) ──\n")
print(comparison.to_string(index=False))

print(f"\n── Summary ──")
print(f"  Random CV tau-b:      {RANDOM_CV_TAU:.3f}")
print(f"  Mean LOFO tau-b:      {mean_tau:.4f}")
print(f"  Mean gap:             {RANDOM_CV_TAU - mean_tau:.4f}")
print(f"  Mean % retained:      {(mean_tau / RANDOM_CV_TAU * 100):.1f}%")

── LOFO tau-b vs Random CV tau-b (0.289) ──

       family  n_devices  lofo_tau_b  p_value  random_cv_tau_b    gap  pct_retained
        FA-Cs         44     -0.0488   0.6416            0.289 0.3378         -16.9
  MA-FA mixed        197     -0.0724   0.1332            0.289 0.3614         -25.1
        Other         88      0.0566   0.4382            0.289 0.2324          19.6
      Pure FA         50      0.1420   0.1476            0.289 0.1470          49.1
      Pure MA        967      0.0783   0.0003            0.289 0.2107          27.1
Triple cation        197     -0.1264   0.0087            0.289 0.4154         -43.7

── Summary ──
  Random CV tau-b:      0.289
  Mean LOFO tau-b:      0.0049
  Mean gap:             0.2841
  Mean % retained:      1.7%


In [5]:
"""Cell 6 — Save results CSV."""

out = comparison.copy()
out.to_csv("Packet_P016_Novel_Composition_Holdout.csv", index=False)
print("Saved: Packet_P016_Novel_Composition_Holdout.csv")
print(f"  Rows: {len(out)}")
print(out.to_string(index=False))

Saved: Packet_P016_Novel_Composition_Holdout.csv
  Rows: 6
       family  n_devices  lofo_tau_b  p_value  random_cv_tau_b    gap  pct_retained
        FA-Cs         44     -0.0488   0.6416            0.289 0.3378         -16.9
  MA-FA mixed        197     -0.0724   0.1332            0.289 0.3614         -25.1
        Other         88      0.0566   0.4382            0.289 0.2324          19.6
      Pure FA         50      0.1420   0.1476            0.289 0.1470          49.1
      Pure MA        967      0.0783   0.0003            0.289 0.2107          27.1
Triple cation        197     -0.1264   0.0087            0.289 0.4154         -43.7


In [6]:
"""Cell 7 — Honest status footer."""

print("=" * 60)
print("P-016  Novel Composition Holdout Simulation")
print("=" * 60)

if mean_tau >= 0.15:
    status = "CONFIRMED"
    verdict = "Model generalises meaningfully to novel composition families."
elif mean_tau >= 0.05:
    status = "PROMISING"
    verdict = "Weak but present ranking signal on novel composition families."
else:
    status = "NEGATIVE"
    verdict = "Model does NOT generalise to novel composition families."

print(f"\n  Status:         {status}")
print(f"  Mean LOFO tau-b: {mean_tau:.4f}")
print(f"  Random CV tau-b: {RANDOM_CV_TAU:.3f}")
print(f"  Verdict:        {verdict}")

print(f"\n  Per-family breakdown:")
for _, r in lofo_df.iterrows():
    flag = "ok" if r["lofo_tau_b"] >= 0.05 else "WEAK"
    print(f"    {r['family']:20s}  tau-b={r['lofo_tau_b']:+.4f}  [{flag}]")

print(f"\n  Decision for lab partners:")
if status == "CONFIRMED":
    print("    We can promise meaningful ranking even on unseen composition families.")
elif status == "PROMISING":
    print("    Ranking on novel families is weak. We should caveat predictions")
    print("    for composition families not in the training data.")
else:
    print("    We CANNOT promise ranking on novel compositions.")
    print("    The model only works within known composition families.")
    print("    Any lab partner bringing a new family needs to be warned.")

print(f"\n  Packet: P-016 | Notebook: 34 | Agent OS v3.0")
print("=" * 60)

P-016  Novel Composition Holdout Simulation

  Status:         NEGATIVE
  Mean LOFO tau-b: 0.0049
  Random CV tau-b: 0.289
  Verdict:        Model does NOT generalise to novel composition families.

  Per-family breakdown:
    FA-Cs                 tau-b=-0.0488  [WEAK]
    MA-FA mixed           tau-b=-0.0724  [WEAK]
    Other                 tau-b=+0.0566  [ok]
    Pure FA               tau-b=+0.1420  [ok]
    Pure MA               tau-b=+0.0783  [ok]
    Triple cation         tau-b=-0.1264  [WEAK]

  Decision for lab partners:
    We CANNOT promise ranking on novel compositions.
    The model only works within known composition families.
    Any lab partner bringing a new family needs to be warned.

  Packet: P-016 | Notebook: 34 | Agent OS v3.0
